# AS02 — LLMBox evaluations on Colab (legal aid clinic scenario)

Backup runner for Part B and Part C when the 16GB Mac is memory-starved. It runs the **same LLMBox code** and the **same open Phi-4-mini-instruct weights**, downloaded to this VM's disk and loaded with `model.source=local`. No external LLM API is called.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then `Runtime → Run all`. Upload `llmbox_colab.zip` when cell 2 asks for it.

**Keep this tab open.** Free Colab keeps running while the tab is open, even if you switch to other windows. It disconnects after about 90 minutes with no interaction, or if you close the tab. Every run writes results per record, and the last cell zips `out/` for download, so a disconnect only loses the run in progress.

T4 does not support bfloat16, so the model loads in **float16**. Say so in the memo.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# 2. Upload llmbox_colab.zip (LLMBox src/ + conf/ + dev/eval sets + schema + citation index)
import os, zipfile
from google.colab import files
os.chdir('/content')
uploaded = files.upload()
zname = next(iter(uploaded))
with zipfile.ZipFile(zname) as z:
    z.extractall('/content')
os.chdir('/content/llmbox-main')
print(os.getcwd()); print(sorted(os.listdir('.')))

Saving llmbox_colab.zip to llmbox_colab (1).zip
/content/llmbox-main
['README.md', '__init__.py', 'about', 'citation_index.json', 'conf', 'data', 'dev_set.jsonl', 'eval_set.jsonl', 'out', 'requirements.txt', 'run_baseline_eval.py', 'run_custom_eval.py', 'schemas', 'src', 'startllm.py', 'structure.py', 'tests', 'transform_original_dataset.py']


In [ ]:
# 3. Dependencies (LLMBox's requirements.txt lists `hydra`; the real PyPI name is `hydra-core`)
!pip -q install -U "transformers>=4.56" hydra-core omegaconf accelerate datasets sentence-transformers fire

In [ ]:
# 4. Download the open Phi-4-mini-instruct weights to local disk (~7.7GB, a few minutes)
from huggingface_hub import snapshot_download
MODEL_PATH = '/content/models/Phi-4-mini-instruct'
snapshot_download('microsoft/Phi-4-mini-instruct', local_dir=MODEL_PATH,
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.py', '*.jinja'])
print(sorted(os.listdir(MODEL_PATH)))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

['.cache', 'added_tokens.json', 'config.json', 'configuration_phi3.py', 'generation_config.json', 'merges.txt', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'modeling_phi3.py', 'sample_finetune.py', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


### Settings
`LIMIT` is how many records each run uses. The assignment asks for 50. Use 15–20 if you are short on time, state the sample size in the memo, and use the same `LIMIT` for every run so the runs stay comparable.

In [ ]:
LIMIT = 50
COMMON = f"--model-local-path {MODEL_PATH} --dtype float16 --limit {LIMIT}"

In [ ]:
%%writefile run_baseline_eval.py
#!/usr/bin/env python3
"""Part B: Baseline LLMBox 'generate' mode evaluation (legal aid clinic
precedent-research task). Extract citation_treatment + cited_cases from an
opinion's raw_text; score against the record's own table_json annotations.

    python run_baseline_eval.py --split dev --run-name eval1 --temperature 1.0 --top-p 0.95 --limit 50
    python run_baseline_eval.py --split dev --run-name eval3 --temperature 0.2 --top-p 0.9  --limit 50
"""
import argparse
import json
import re
import time
from collections import Counter
from pathlib import Path

from hydra import initialize, compose
from src.schema import register_configs
from src.generation import GenerationManager

VALID_TREATMENTS = [
    "cited", "referred to", "applied", "followed", "considered", "discussed",
    "distinguished", "quoted", "related", "affirmed", "approved",
]

SYSTEM_PROMPT = (
    "You are assisting a legal aid clinic's precedent-research task. You will "
    "be shown the full text of an Australian Federal Court opinion. Read it "
    "and answer in exactly this format, with no other text:\n\n"
    "CITATION TREATMENT: <one of: cited, referred to, applied, followed, "
    "considered, discussed, distinguished, quoted, related, affirmed, approved>\n"
    "CITED CASES: <semicolon-separated list of case names/citations exactly as "
    "they appear in the opinion, or 'none' if there are none>"
)


def parse_response(text):
    treatment = None
    m = re.search(r"CITATION TREATMENT:\s*(.+)", text, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip().strip(".").lower()
        for v in VALID_TREATMENTS:
            if v in candidate:
                treatment = v
                break
    cases = []
    m = re.search(r"CITED CASES:\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if m:
        raw = m.group(1).strip()
        if raw.lower() not in ("none", "none.", ""):
            parts = raw.split(";") if ";" in raw else raw.split("\n")
            cases = [p.strip().strip(".").lstrip("-* ").strip() for p in parts if p.strip()]
    return treatment, cases


def gold_for(record):
    table = record.get("table_json") or []
    classes = Counter(c["citation_class"] for c in table if c.get("citation_class"))
    gold_treatment = classes.most_common(1)[0][0] if classes else None
    return gold_treatment, [c["cited_case"] for c in table]


def score_cases(pred, gold):
    pred_set = {p.strip().lower() for p in pred}
    gold_set = {g.strip().lower() for g in gold}
    if not pred_set and not gold_set:
        return 1.0, 1.0, 1.0
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def count_tokens(tokenizer, messages, response):
    in_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    if isinstance(in_ids, dict):
        in_ids = in_ids["input_ids"]
    out_ids = tokenizer(response, add_special_tokens=False)["input_ids"]
    return len(in_ids), len(out_ids)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", choices=["dev", "eval"], required=True)
    ap.add_argument("--run-name", required=True)
    ap.add_argument("--model-local-path", default="/content/models/Phi-4-mini-instruct")
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--temperature", type=float, default=1.0)
    ap.add_argument("--top-p", type=float, default=0.95)
    ap.add_argument("--max-new-tokens", type=int, default=512)
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    records = [json.loads(l) for l in open(f"{args.split}_set.jsonl", encoding="utf-8")]
    if args.limit:
        records = records[:args.limit]
    print(f"Loaded {len(records)} records from {args.split}_set.jsonl", flush=True)

    register_configs()
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name="config", overrides=[
            "model=phi4_instruct", "mode=generate",
            f"model.local_path={args.model_local_path}",
            f"model.dtype={args.dtype}",
            f"generation.temperature={args.temperature}",
            f"generation.top_p={args.top_p}",
            f"generation.max_new_tokens={args.max_new_tokens}",
        ])
    cfg.system_prompt = SYSTEM_PROMPT

    gen = GenerationManager()
    model, tokenizer, device = gen._load_model_and_tokenizer(cfg)
    print(f"Model on {device}. temperature={args.temperature} top_p={args.top_p} "
          f"max_new_tokens={args.max_new_tokens}", flush=True)

    Path("out").mkdir(exist_ok=True)
    out_path = Path("out") / f"{args.run_name}_responses.jsonl"
    rows = []
    t_start = time.time()
    with open(out_path, "w", encoding="utf-8") as outf:
        for i, record in enumerate(records):
            messages = [
                {"role": "system", "content": cfg.system_prompt},
                {"role": "user", "content": f"Opinion:\n{record['raw_text']}"},
            ]
            t0 = time.time()
            response = gen._generate_once(model, tokenizer, device, messages, cfg)
            latency = time.time() - t0
            in_tok, out_tok = count_tokens(tokenizer, messages, response)
            pred_treatment, pred_cases = parse_response(response)
            gold_treatment, gold_cases = gold_for(record)
            precision, recall, f1 = score_cases(pred_cases, gold_cases)
            row = {
                "doc_id": record["doc_id"], "response": response,
                "pred_treatment": pred_treatment, "pred_cases": pred_cases,
                "gold_treatment": gold_treatment, "gold_cases": gold_cases,
                "treatment_correct": (pred_treatment == gold_treatment) if gold_treatment else None,
                "cases_precision": precision, "cases_recall": recall, "cases_f1": f1,
                "parse_failed": pred_treatment is None,
                "input_tokens": in_tok, "output_tokens": out_tok,
                "latency_s": round(latency, 2),
            }
            rows.append(row)
            outf.write(json.dumps(row, ensure_ascii=False) + "\n")
            outf.flush()
            print(f"  [{i+1}/{len(records)}] {record['doc_id']}  treatment={pred_treatment} "
                  f"(gold {gold_treatment})  f1={f1:.2f}  {latency:.1f}s", flush=True)

    n = len(rows)
    scored = [r for r in rows if r["gold_treatment"]]
    summary = {
        "run_name": args.run_name, "split": args.split, "n_records": n,
        "hyperparameters": {"temperature": args.temperature, "top_p": args.top_p,
                            "max_new_tokens": args.max_new_tokens},
        "citation_treatment_accuracy": round(sum(r["treatment_correct"] for r in scored) / len(scored), 3) if scored else None,
        "cited_cases_precision": round(sum(r["cases_precision"] for r in rows) / n, 3),
        "cited_cases_recall": round(sum(r["cases_recall"] for r in rows) / n, 3),
        "cited_cases_f1": round(sum(r["cases_f1"] for r in rows) / n, 3),
        "parse_failure_rate": round(sum(r["parse_failed"] for r in rows) / n, 3),
        "mean_input_tokens": round(sum(r["input_tokens"] for r in rows) / n, 1),
        "mean_output_tokens": round(sum(r["output_tokens"] for r in rows) / n, 1),
        "mean_latency_s": round(sum(r["latency_s"] for r in rows) / n, 2),
        "total_time_s": round(time.time() - t_start, 1),
        "device": str(device),
    }
    (Path("out") / f"{args.run_name}_summary.json").write_text(json.dumps(summary, indent=2))
    print("\nSUMMARY " + json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting run_baseline_eval.py


In [ ]:
%%writefile run_custom_eval.py
#!/usr/bin/env python3
"""Part C: custom LLMBox evaluations for the legal aid clinic scenario.

  structured_output -- case-intake extraction constrained to the Pydantic-
      derived JSON schema in schemas/case_intake.json. Scored on schema
      validity, citation_treatment accuracy, cited_cases P/R/F1.

  tool_calling -- the model names the main case an opinion cites and must
      check it with verify_citation (src/tools.py) before answering: the
      clinic's "catch a hallucinated citation before it is filed" task.
      Scored on tool-call rate, whether the checked case is real (in the
      opinion's own table_json), and whether the model's final answer
      agrees with what the tool actually returned.

    python run_custom_eval.py --task structured_output --limit 50
    python run_custom_eval.py --task tool_calling --limit 50
"""
import argparse
import json
import re
import time
from collections import Counter
from pathlib import Path

from hydra import initialize, compose
from src.schema import register_configs
from src.generation import GenerationManager

STRUCT_SYSTEM_PROMPT = (
    "You are assisting a legal aid clinic's case-intake process. You will be "
    "shown the full text of an Australian Federal Court opinion. Extract the "
    "visa category, how the opinion treats the main precedent it cites, and "
    "the cases it cites."
)

TOOL_SYSTEM_PROMPT = (
    "You are assisting a legal aid clinic's precedent-research task. Before "
    "any case citation goes into a brief, the clinic requires you to verify it "
    "with the verify_citation tool. Read the opinion, identify the single most "
    "important case it cites, call verify_citation with that exact case name, "
    "and then reply with exactly two lines:\n"
    "CASE: <the case name you checked>\n"
    "VERIFIED: <true or false, exactly as the tool reported>"
)

TOOLS = [{
    "name": "verify_citation",
    "description": "Check whether a case citation is a real, known citation before including it in a legal brief.",
    "parameters": {
        "type": "object",
        "properties": {"case_name": {"type": "string",
                                     "description": "The case name/citation to verify, exactly as it appears in the opinion."}},
        "required": ["case_name"],
    },
}]


def gold_for(record):
    table = record.get("table_json") or []
    classes = Counter(c["citation_class"] for c in table if c.get("citation_class"))
    gold_treatment = classes.most_common(1)[0][0] if classes else None
    return gold_treatment, [c["cited_case"] for c in table]


def score_cases(pred, gold):
    pred_set = {str(p).strip().lower() for p in pred}
    gold_set = {g.strip().lower() for g in gold}
    if not pred_set and not gold_set:
        return 1.0, 1.0, 1.0
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def extract_json(text):
    """Strict first (the API contract), then a lenient fallback that strips a
    ```json fence -- reported separately so the memo can show both."""
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0)), False
        except json.JSONDecodeError:
            pass
    return None, False


def count_tokens(tokenizer, messages, response):
    in_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    if isinstance(in_ids, dict):
        in_ids = in_ids["input_ids"]
    return len(in_ids), len(tokenizer(response, add_special_tokens=False)["input_ids"])


def run_structured_output(args, records, gen, model, tokenizer, device, cfg):
    schema_text = Path(args.schema_path).read_text(encoding="utf-8")
    system = (STRUCT_SYSTEM_PROMPT + "\n\nRespond with ONLY a single JSON object that strictly "
              f"matches this JSON Schema, with no other text:\n{schema_text}")
    rows = []
    for i, record in enumerate(records):
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": f"Opinion:\n{record['raw_text']}"}]
        t0 = time.time()
        response = gen._generate_once(model, tokenizer, device, messages, cfg)
        latency = time.time() - t0
        in_tok, out_tok = count_tokens(tokenizer, messages, response)
        gold_treatment, gold_cases = gold_for(record)
        parsed, strict_ok = extract_json(response)
        required = ("visa_category", "citation_treatment", "cited_cases")
        schema_valid = isinstance(parsed, dict) and all(k in parsed for k in required)
        echoed_schema = isinstance(parsed, dict) and "properties" in parsed and "visa_category" not in parsed
        pred_treatment = parsed.get("citation_treatment") if schema_valid else None
        pred_cases = (parsed.get("cited_cases") or []) if schema_valid else []
        if not isinstance(pred_cases, list):
            pred_cases = [pred_cases]
        precision, recall, f1 = score_cases(pred_cases, gold_cases)
        row = {
            "doc_id": record["doc_id"], "response": response,
            "strict_json": strict_ok, "schema_valid": schema_valid, "echoed_schema": echoed_schema,
            "pred_visa_category": parsed.get("visa_category") if schema_valid else None,
            "pred_treatment": pred_treatment, "gold_treatment": gold_treatment,
            "treatment_correct": (pred_treatment == gold_treatment) if gold_treatment else None,
            "cases_precision": precision, "cases_recall": recall, "cases_f1": f1,
            "input_tokens": in_tok, "output_tokens": out_tok, "latency_s": round(latency, 2),
        }
        rows.append(row)
        print(f"  [{i+1}/{len(records)}] {record['doc_id']}  valid={schema_valid} strict={strict_ok} "
              f"echo={echoed_schema} treatment={pred_treatment} f1={f1:.2f} {latency:.1f}s", flush=True)
    return rows


def run_tool_calling(args, records, gen, model, tokenizer, device, cfg):
    rows = []
    for i, record in enumerate(records):
        _, gold_cases = gold_for(record)
        gold_set = {g.strip().lower() for g in gold_cases}
        messages = [{"role": "system", "content": gen.build_functools_system_prompt(TOOL_SYSTEM_PROMPT, TOOLS)},
                    {"role": "user", "content": f"Opinion:\n{record['raw_text']}"}]
        first_turn = list(messages)
        t0 = time.time()
        answer, calls = gen.run_tool_turn(model, tokenizer, device, messages, cfg)
        latency = time.time() - t0
        in_tok, out_tok = count_tokens(tokenizer, first_turn, answer)
        tool_called = len(calls) > 0
        checked_case = (calls[0].get("arguments") or {}).get("case_name") if tool_called else None
        tool_verified = (calls[0].get("result") or {}).get("verified") if tool_called else None
        case_is_real = (str(checked_case).strip().lower() in gold_set) if checked_case else None
        m = re.search(r"VERIFIED:\s*(true|false)", answer, re.IGNORECASE)
        claimed = (m.group(1).lower() == "true") if m else None
        consistent = (claimed == tool_verified) if (claimed is not None and tool_verified is not None) else None
        row = {
            "doc_id": record["doc_id"], "answer": answer, "tool_calls": calls,
            "tool_called": tool_called, "checked_case": checked_case,
            "case_in_opinion_table": case_is_real, "tool_verified": tool_verified,
            "model_claimed_verified": claimed, "model_consistent_with_tool": consistent,
            "claimed_verified_without_tool": (claimed is True and not tool_called),
            "input_tokens": in_tok, "output_tokens": out_tok, "latency_s": round(latency, 2),
        }
        rows.append(row)
        print(f"  [{i+1}/{len(records)}] {record['doc_id']}  tool={tool_called} case={str(checked_case)[:50]!r} "
              f"verified={tool_verified} claimed={claimed} {latency:.1f}s", flush=True)
    return rows


def rate(rows, key):
    vals = [r[key] for r in rows if r[key] is not None]
    return round(sum(vals) / len(vals), 3) if vals else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["structured_output", "tool_calling"], required=True)
    ap.add_argument("--split", choices=["dev", "eval"], default="eval")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--model-local-path", default="/content/models/Phi-4-mini-instruct")
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--schema-path", default="schemas/case_intake.json")
    ap.add_argument("--temperature", type=float, default=0.2)
    ap.add_argument("--top-p", type=float, default=0.9)
    ap.add_argument("--max-new-tokens", type=int, default=512)
    args = ap.parse_args()

    records = [json.loads(l) for l in open(f"{args.split}_set.jsonl", encoding="utf-8")]
    if args.limit:
        records = records[:args.limit]
    print(f"Loaded {len(records)} records from {args.split}_set.jsonl", flush=True)

    register_configs()
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name="config", overrides=[
            "model=phi4_instruct", f"mode={args.task}",
            f"model.local_path={args.model_local_path}",
            f"model.dtype={args.dtype}",
            f"generation.temperature={args.temperature}",
            f"generation.top_p={args.top_p}",
            f"generation.max_new_tokens={args.max_new_tokens}",
        ])
    gen = GenerationManager()
    model, tokenizer, device = gen._load_model_and_tokenizer(cfg)
    print(f"Model on {device}. task={args.task}", flush=True)

    t_start = time.time()
    run = run_structured_output if args.task == "structured_output" else run_tool_calling
    rows = run(args, records, gen, model, tokenizer, device, cfg)

    Path("out").mkdir(exist_ok=True)
    with open(Path("out") / f"{args.task}_responses.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

    n = len(rows)
    summary = {"task": args.task, "split": args.split, "n_records": n,
               "hyperparameters": {"temperature": args.temperature, "top_p": args.top_p,
                                   "max_new_tokens": args.max_new_tokens}}
    if args.task == "structured_output":
        summary.update({
            "strict_json_rate": rate(rows, "strict_json"),
            "schema_valid_rate": rate(rows, "schema_valid"),
            "schema_echo_rate": rate(rows, "echoed_schema"),
            "citation_treatment_accuracy": rate(rows, "treatment_correct"),
            "cited_cases_precision": round(sum(r["cases_precision"] for r in rows) / n, 3),
            "cited_cases_recall": round(sum(r["cases_recall"] for r in rows) / n, 3),
            "cited_cases_f1": round(sum(r["cases_f1"] for r in rows) / n, 3),
        })
    else:
        summary.update({
            "tool_call_rate": rate(rows, "tool_called"),
            "checked_case_in_opinion_rate": rate(rows, "case_in_opinion_table"),
            "tool_verified_rate": rate(rows, "tool_verified"),
            "model_consistent_with_tool_rate": rate(rows, "model_consistent_with_tool"),
            "claimed_verified_without_tool_rate": rate(rows, "claimed_verified_without_tool"),
        })
    summary.update({
        "mean_input_tokens": round(sum(r["input_tokens"] for r in rows) / n, 1),
        "mean_output_tokens": round(sum(r["output_tokens"] for r in rows) / n, 1),
        "mean_latency_s": round(sum(r["latency_s"] for r in rows) / n, 2),
        "total_time_s": round(time.time() - t_start, 1),
        "device": str(device),
    })
    (Path("out") / f"{args.task}_summary.json").write_text(json.dumps(summary, indent=2))
    print("\nSUMMARY " + json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()


Overwriting run_custom_eval.py


In [ ]:
# Register the verify_citation tool in LLMBox's tool registry (skipped if already there)
TOOL_PATCH = '\n# --------------------------------------------------------------------------\n# verify_citation -- legal aid clinic tool: check a case citation against the\n# known-citation index (built from the corpus\'s own table_json) before it\n# goes into a filed brief. Exact match, case-insensitive; can confirm, never\n# deny (absence from the index is not proof a citation is fake).\n# --------------------------------------------------------------------------\nimport json as _json\nfrom pathlib import Path as _Path\n\n_CITATION_INDEX_PATH = _Path(__file__).resolve().parent.parent / "citation_index.json"\ntry:\n    _CITATION_INDEX = _json.loads(_CITATION_INDEX_PATH.read_text(encoding="utf-8"))\nexcept FileNotFoundError:\n    _CITATION_INDEX = {}\n_CITATION_LOOKUP = {k.strip().lower(): (k, v) for k, v in _CITATION_INDEX.items()}\n\n\n@register_tool("verify_citation")\ndef verify_citation(case_name):\n    if not isinstance(case_name, str) or not case_name.strip():\n        return {"verified": False, "error": "case_name must be a non-empty string"}\n    hit = _CITATION_LOOKUP.get(case_name.strip().lower())\n    if hit:\n        return {"verified": True, "matched_name": hit[0],\n                "citation_class_in_corpus": hit[1].get("citation_class")}\n    return {"verified": False, "matched_name": None}\n'
src = open('src/tools.py').read()
if 'register_tool("verify_citation")' not in src:
    open('src/tools.py', 'a').write(TOOL_PATCH)
    print('added verify_citation to src/tools.py')
else:
    print('verify_citation already registered')
import json
print(len(json.load(open('citation_index.json'))), 'citations in index')

verify_citation already registered
876 citations in index


In [ ]:
# PATCH: cap input length + survive GPU out-of-memory. Run once, then re-run the eval cells.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MAX_INPUT_TOKENS"] = "6000"

HELPERS = '''
import os as _os
MAX_INPUT_TOKENS = int(_os.environ.get("MAX_INPUT_TOKENS", "6000"))

def truncate_text(tokenizer, text, max_tokens=None):
    max_tokens = max_tokens or MAX_INPUT_TOKENS
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens])

def make_oom_safe(gen):
    import torch
    orig = gen._generate_once
    def safe(*a, **k):
        try:
            return orig(*a, **k)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            return "[OOM]"
    gen._generate_once = safe
'''

for path in ["run_baseline_eval.py", "run_custom_eval.py"]:
    src = open(path).read()
    if "MAX_INPUT_TOKENS" in src:
        print(path, "already patched"); continue
    src = src.replace("from src.generation import GenerationManager\n",
                      "from src.generation import GenerationManager\n" + HELPERS, 1)
    src = src.replace("f\"Opinion:\\n{record['raw_text']}\"",
                      "f\"Opinion:\\n{truncate_text(tokenizer, record['raw_text'])}\"")
    src = src.replace("    gen = GenerationManager()\n",
                      "    gen = GenerationManager()\n    make_oom_safe(gen)\n", 1)
    compile(src, path, "exec")
    open(path, "w").write(src)
    print(path, "patched:", src.count("truncate_text(tokenizer"), "truncation point(s), OOM guard on")

run_baseline_eval.py patched: 2 truncation point(s), OOM guard on
run_custom_eval.py patched: 3 truncation point(s), OOM guard on


## Part B — Baseline Evaluation 1 (LLMBox defaults: temperature 1.0, top_p 0.95, max_new_tokens 512)

In [ ]:
!python run_baseline_eval.py --split dev --run-name eval1 --temperature 1.0 --top-p 0.95 --max-new-tokens 512 {COMMON}

Loaded 50 records from dev_set.jsonl
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100% 194/194 [00:30<00:00,  6.30it/s]
Model on cuda. temperature=1.0 top_p=0.95 max_new_tokens=512
  [1/50] legal_06_280  treatment=cited (gold cited)  f1=0.33  5.9s
  [2/50] legal_07_391  treatment=None (gold followed)  f1=0.00  19.0s
  [3/50] legal_09_912  treatment=None (gold applied)  f1=0.00  0.4s
  [4/50] legal_06_1275  treatment=None (gold referred to)  f1=0.00  0.5s
  [5/50] legal_09_392  treatment=None (gold applied)  f1=1.00  14.4s
  [6/50] legal_06_1615  treatment=referred to (gold applied)  f1=0.30  21.2s
  [7/50] legal_07_723  treatment=

## Part B — Baseline Evaluation 3 (near-greedy: temperature 0.2, top_p 0.9, max_new_tokens 512)

In [ ]:
!python run_baseline_eval.py --split dev --run-name eval3 --temperature 0.2 --top-p 0.9 --max-new-tokens 512 {COMMON}

Loaded 50 records from dev_set.jsonl
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100% 194/194 [00:30<00:00,  6.29it/s]
Model on cuda. temperature=0.2 top_p=0.9 max_new_tokens=512
  [1/50] legal_06_280  treatment=referred to (gold cited)  f1=0.50  4.7s
  [2/50] legal_07_391  treatment=None (gold followed)  f1=0.00  63.3s
  [3/50] legal_09_912  treatment=None (gold applied)  f1=0.00  0.4s
  [4/50] legal_06_1275  treatment=None (gold referred to)  f1=0.00  0.4s
  [5/50] legal_09_392  treatment=None (gold applied)  f1=1.00  15.7s
  [6/50] legal_06_1615  treatment=referred to (gold applied)  f1=0.00  4.8s
  [7/50] legal_07_723  treatm

## Part C — Custom Evaluation 1: generate + structured_output (case-intake schema, eval split)

In [ ]:
!python run_custom_eval.py --task structured_output --split eval {COMMON}

Loaded 50 records from eval_set.jsonl
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100% 194/194 [00:30<00:00,  6.26it/s]
Model on cuda. task=structured_output
  [1/50] legal_07_436  valid=False strict=False echo=False treatment=None f1=0.00 2.0s
  [2/50] legal_06_650  valid=True strict=False echo=False treatment=followed f1=0.75 12.2s
  [3/50] legal_08_91  valid=False strict=False echo=False treatment=None f1=0.00 0.4s
  [4/50] legal_07_1037  valid=True strict=False echo=False treatment=followed f1=0.40 9.5s
  [5/50] legal_06_886  valid=False strict=False echo=False treatment=None f1=0.00 24.0s
  [6/50] legal_06_1166  valid=True s

## Part C — Custom Evaluation 2: generate + tool_calling (verify_citation, eval split)

In [ ]:
!python run_custom_eval.py --task tool_calling --split eval {COMMON}

Loaded 50 records from eval_set.jsonl
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100% 194/194 [00:30<00:00,  6.27it/s]
Model on cuda. task=tool_calling
  [1/50] legal_07_436  tool=False case='None' verified=None claimed=None 2.0s
  [2/50] legal_06_650  tool=False case='None' verified=None claimed=True 3.0s
  [3/50] legal_08_91  tool=False case='None' verified=None claimed=None 0.4s
  [4/50] legal_07_1037  tool=False case='None' verified=None claimed=True 3.1s
  [5/50] legal_06_886  tool=False case='None' verified=None claimed=True 7.1s
  [6/50] legal_06_1166  tool=False case='None' verified=None claimed=True 4.6s
  [7/50] legal_

## Results

In [ ]:
import glob, json
for p in sorted(glob.glob('out/*_summary.json')):
    print('=' * 60); print(p); print(json.dumps(json.load(open(p)), indent=2))

out/eval1_summary.json
{
  "run_name": "eval1",
  "split": "dev",
  "n_records": 50,
  "hyperparameters": {
    "temperature": 1.0,
    "top_p": 0.95,
    "max_new_tokens": 512
  },
  "citation_treatment_accuracy": 0.08,
  "cited_cases_precision": 0.105,
  "cited_cases_recall": 0.161,
  "cited_cases_f1": 0.12,
  "parse_failure_rate": 0.56,
  "mean_input_tokens": 2.0,
  "mean_output_tokens": 79.7,
  "mean_latency_s": 10.26,
  "total_time_s": 515.1,
  "device": "cuda"
}
out/eval3_summary.json
{
  "run_name": "eval3",
  "split": "dev",
  "n_records": 50,
  "hyperparameters": {
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 512
  },
  "citation_treatment_accuracy": 0.04,
  "cited_cases_precision": 0.116,
  "cited_cases_recall": 0.187,
  "cited_cases_f1": 0.138,
  "parse_failure_rate": 0.56,
  "mean_input_tokens": 2.0,
  "mean_output_tokens": 148.9,
  "mean_latency_s": 17.31,
  "total_time_s": 867.2,
  "device": "cuda"
}
out/structured_output_summary.json
{
  "task": "struc

In [ ]:
# Zip everything in out/ and download it
!zip -qr /content/as02_outputs.zip out
files.download('/content/as02_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python run_round2.py --limit 15

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100% 194/194 [00:31<00:00,  6.21it/s]
model on cuda
TOKEN STATS {
  "dev": {
    "n": 50,
    "mean_opinion_tokens": 5288.4,
    "median_opinion_tokens": 4468,
    "max_opinion_tokens": 23375,
    "share_over_6000": 0.26,
    "share_over_3000": 0.64,
    "mean_tokens_sent_at_cap_6000": 4142.9,
    "mean_tokens_sent_at_cap_3000": 2741.7
  },
  "eval": {
    "n": 50,
    "mean_opinion_tokens": 5576.4,
    "median_opinion_tokens": 4320,
    "max_opinion_tokens": 19961,
    "share_over_6000": 0.3,
    "share_over_3000": 0.8,
    "mean_tokens_sent_at_cap_6000": 4313.6,
    "mean_tokens_sent_